In [0]:
from pyspark.sql.functions import col, when, timestamp_diff, max, min
from datetime import date
from dateutil.relativedelta import relativedelta

In [0]:
# Get the first day of the month 2 months ago
two_months_ago_start = date.today().replace(day=1) - relativedelta(months=3)

# Get the first day of the month 1 month ago
one_month_ago_start = date.today().replace(day=1) - relativedelta(months=1)

In [0]:
# Read the 'yellow_trips_raw' table from the 'nyctaxi.01_bronze' schema
# Then filter rows where 'tpep_pickup_datetime' is >= two_months_ago_start
# and < one_month_ago_start (i.e., only the month that is two months before today)
df = spark.read.table("yuli_demo_nyctaxi.01_bronze.yellow_trips_raw").filter(
    f"tpep_pickup_datetime >= '{two_months_ago_start}' AND tpep_pickup_datetime < '{one_month_ago_start}'"
)

##### Verify once again that all data are within the scope of last 2 months

In [0]:
df.agg(max("tpep_pickup_datetime"),min("tpep_pickup_datetime")).display()

max(tpep_pickup_datetime),min(tpep_pickup_datetime)
2026-08-31T23:59:59.000,2026-01-01T00:00:00.000


In [0]:
df = df.select(
    when(col("VendorID") == 1, "Creative Mobile Technologies, LLC")
    .when(col("VendorID") == 2, "Curb Mobility, LLC")
    .when(col("VendorID") == 6, "Myle Technologies Inc")
    .when(col("VendorID") == 7, "Helix")
    .otherwise("Unknown")
    .alias("vendor"),
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    timestamp_diff("MINUTE", df.tpep_pickup_datetime, df.tpep_dropoff_datetime).alias("trip_duration"),
    "passenger_count",
    "trip_distance",
    when(col("RatecodeID") == 1, "Standard Rate")
    .when(col("RatecodeID") == 2, "JFK")
    .when(col("RatecodeID") == 3, "Newark")
    .when(col("RatecodeID") == 4, "Nassau or Westchester")
    .when(col("RatecodeID") == 5, "Negotiated Fare")
    .when(col("RatecodeID") == 6, "Group Ride")
    .otherwise("Unknown")
    .alias("rate_type"),
    "store_and_fwd_flag",
    col("PULocationID").alias("pu_location_id"),
    col("DOLocationID").alias("do_location_id"),
    when(col("payment_type") == 0, "Flex Fare trip")
    .when(col("payment_type") == 1, "Credit card")
    .when(col("payment_type") == 2, "Cash")
    .when(col("payment_type") == 3, "No charge")
    .when(col("payment_type") == 4, "Dispute")
    .when(col("payment_type") == 6, "Voided trip")
    .otherwise("Unknown")
    .alias("payment_type"),
    
    "fare_amount",
    "extra",
    "mta_tax",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "congestion_surcharge",
    col("Airport_fee").alias("airport_fee"),
    col("cbd_congestion_fee"),
    "processed_timestamp"
)

In [0]:
# Write the Spark DataFrame to the Silver table using append mode
df.write.mode("append").saveAsTable("yuli_demo_nyctaxi.02_silver.yellow_trips_cleansed")